# Очистка данных Online Retail II

В этом ноутбуке я обработаю найденные проблемы качества данных и подготовлю таблицу к дальнейшему анализу продаж и покупателей.

Исходный набор данных сохраняется без изменений.

## Загрузка данных

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
df = pd.read_pickle('online_retail_raw.pkl')
df_clean = df.copy()

df_clean.shape

(1067371, 9)

## Типы данных

Приведём номера счетов и коды товаров к строковому типу, а идентификатор покупателя — к целому типу с поддержкой пропусков.

In [3]:
df_clean['invoice'] = df_clean['invoice'].astype(str)
df_clean['stock_code'] = df_clean['stock_code'].astype(str)
df_clean['customer_id'] = df_clean['customer_id'].astype('Int64')
df_clean['invoice_date'] = pd.to_datetime(df_clean['invoice_date'])

In [4]:
df_clean.dtypes

invoice                  object
stock_code               object
description              object
quantity                  int64
invoice_date     datetime64[ns]
price                   float64
customer_id               Int64
country                  object
source_period            object
dtype: object

У текстовых полей удалим пробелы в начале и конце строк.

In [5]:
df_clean['stock_code'] = df_clean['stock_code'].str.strip()
df_clean['description'] = df_clean['description'].str.strip()
df_clean['country'] = df_clean['country'].str.strip()

## Дубликаты

В исходных данных были найдены полностью совпадающие строки. Для аналитической выборки удалим их как вероятные повторы загрузки. Это является допущением проекта, поскольку отдельного идентификатора строки в исходных данных нет.

In [6]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
duplicates_removed = rows_before - len(df_clean)

print('Строк до удаления:', rows_before)
print('Удалено дубликатов:', duplicates_removed)
print('Строк после удаления:', len(df_clean))

Строк до удаления: 1067371
Удалено дубликатов: 12133
Строк после удаления: 1055238


## Отмены и отрицательное количество

Создадим отдельные признаки для счетов с префиксом `C` и строк с отрицательным количеством.

In [7]:
df_clean['is_cancelled'] = df_clean['invoice'].str.startswith('C')
df_clean['is_negative_quantity'] = df_clean['quantity'] < 0

In [8]:
pd.crosstab(df_clean['is_negative_quantity'], df_clean['is_cancelled'])

is_cancelled,False,True
is_negative_quantity,,
False,1032348,1
True,3457,19432


Большая часть строк с отрицательным количеством относится к отменённым счетам, но встречаются и строки без префикса `C`.

Поэтому отрицательное количество будем хранить как отдельный признак и не считать такие строки обычными продажами.

## Стоимость позиции

Рассчитаем стоимость каждой строки как произведение количества товара на цену.

In [9]:
df_clean['line_total'] = df_clean['quantity'] * df_clean['price']

In [10]:
df_clean[['quantity', 'price', 'line_total']].describe().T

,count,mean,std,min,25%,50%,75%,max
quantity,"1,055,238.00",10.02,173.68,"-80,995.00",1.00,3.00,10.00,"80,995.00"
price,"1,055,238.00",4.67,124.26,"-53,594.36",1.25,2.10,4.15,"38,970.00"
line_total,"1,055,238.00",18.23,294.09,"-168,469.60",3.75,9.90,17.70,"168,469.60"


У отменённых операций стоимость позиции получается отрицательной. Такие строки нельзя включать в обычную выручку, но их можно использовать для отдельного анализа возвратов.

## Строки с некорректной ценой

Проверим строки с нулевой и отрицательной ценой.

In [11]:
df_clean[df_clean['price'] <= 0][['invoice', 'stock_code', 'description', 'quantity', 'price', 'customer_id']].head(20)

,invoice,stock_code,description,quantity,price,customer_id
263,489464,21733,85123a mixed,-96,0.00,<NA>
283,489463,71477,short,-240,0.00,<NA>
284,489467,85123A,21733 mixed,-192,0.00,<NA>
470,489521,21646,NaN,-50,0.00,<NA>
3114,489655,20683,NaN,-44,0.00,<NA>
3161,489659,21350,NaN,230,0.00,<NA>
3162,489660,35956,lost,-1043,0.00,<NA>
3168,489663,35605A,damages,-117,0.00,<NA>
3731,489781,84292,NaN,17,0.00,<NA>
4296,489806,18010,NaN,-770,0.00,<NA>


In [12]:
price_checks = pd.Series({'negative_price': (df_clean['price'] < 0).sum(), 'zero_price': (df_clean['price'] == 0).sum()})
price_checks

negative_price       5
zero_price        6191
dtype: int64

Строки с отрицательной ценой не подходят для расчёта продаж. Нулевая цена может означать бесплатный товар, подарок или служебную операцию.

Для расчёта выручки будем использовать только строки с положительной ценой.

## Пропуски

Строки без `customer_id` можно учитывать при анализе общей выручки, но нельзя использовать в анализе поведения покупателей.

In [13]:
print('Без customer_id:', df_clean['customer_id'].isna().sum())
print('Без description:', df_clean['description'].isna().sum())

Без customer_id: 242870
Без description: 4386


Удалять все строки без покупателя из основной таблицы не будем. Вместо этого позднее создадим отдельную выборку для когортного анализа и RFM.

## Служебные операции

В таблице встречаются почтовые расходы, комиссии, ручные корректировки и другие строки, которые не являются продажами товаров. Сохраним их в общей таблице, но исключим из выборки товарных продаж.

In [14]:
service_codes = ['M', 'POST', 'DOT', 'C2', 'D', 'BANK CHARGES', 'AMAZONFEE', 'B', 'CRUK', 'S', 'PADS', 'TEST001', 'TEST002']

df_clean['is_service'] = df_clean['stock_code'].isin(service_codes)

df_clean[df_clean['is_service']][['stock_code', 'description']].drop_duplicates().sort_values('stock_code')

,stock_code,description
440688,AMAZONFEE,AMAZON FEE
179403,B,Adjust bad debt
18410,BANK CHARGES,Bank Charges
9292,C2,CARRIAGE
32207,C2,NaN
842969,CRUK,CRUK Commission
735,D,Discount
2379,DOT,DOTCOM POSTAGE
271469,DOT,NaN
2697,M,Manual


## Выборка обычных продаж

Для анализа продаж оставим строки, которые одновременно удовлетворяют условиям:

- счёт не отменён;
- строка не является служебной операцией;
- количество положительное;
- цена положительная.

In [15]:
sales = df_clean[(~df_clean['is_cancelled']) & (~df_clean['is_service']) & (df_clean['quantity'] > 0) & (df_clean['price'] > 0)].copy()

sales.shape

(1025074, 13)

In [16]:
assert sales['is_cancelled'].sum() == 0
assert sales['is_service'].sum() == 0
assert (sales['quantity'] > 0).all()
assert (sales['price'] > 0).all()

print('Проверки пройдены')

Проверки пройдены


Для анализа покупателей дополнительно нужны строки с заполненным `customer_id`.

In [17]:
customer_sales = sales[sales['customer_id'].notna()].copy()

customer_sales.shape

(790739, 13)

## Проверка результата

In [18]:
cleaning_result = pd.Series({'raw_rows': len(df), 'clean_rows': len(df_clean), 'sales_rows': len(sales), 'customer_sales_rows': len(customer_sales), 'duplicates_removed': duplicates_removed, 'cancelled_rows': df_clean['is_cancelled'].sum(), 'negative_quantity_rows': df_clean['is_negative_quantity'].sum(), 'service_rows': df_clean['is_service'].sum()})

cleaning_result

raw_rows                  1067371
clean_rows                1055238
sales_rows                1025074
customer_sales_rows        790739
duplicates_removed          12133
cancelled_rows              19433
negative_quantity_rows      22889
service_rows                 5743
dtype: int64

In [19]:
sales[['quantity', 'price', 'line_total']].describe().T

,count,mean,std,min,25%,50%,75%,max
quantity,"1,025,074.00",11.09,127.50,1.00,1.00,4.00,12.00,"80,995.00"
price,"1,025,074.00",3.36,7.00,0.03,1.25,2.10,4.13,"5,117.03"
line_total,"1,025,074.00",19.57,198.23,0.06,4.13,10.04,17.70,"168,469.60"


## Проверка больших значений

Очень крупные количества и суммы не будем автоматически удалять. Сначала посмотрим верхние квантили и несколько самых крупных операций.

In [20]:
quantiles = sales[['quantity', 'price', 'line_total']].quantile([0.5, 0.9, 0.95, 0.99, 0.999])
quantiles.index = ['50%', '90%', '95%', '99%', '99.9%']

quantiles

,quantity,price,line_total
50%,4.00,2.10,10.04
90%,24.00,7.62,32.85
95%,30.00,9.95,59.50
99%,104.00,16.95,179.00
99.9%,500.00,35.75,765.00


In [21]:
sales.nlargest(15, 'line_total')[['invoice', 'stock_code', 'description', 'quantity', 'price', 'line_total', 'customer_id', 'country']]

,invoice,stock_code,description,quantity,price,line_total,customer_id,country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,"168,469.60",16446,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,"77,183.60",12346,United Kingdom
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,"38,970.00",15098,United Kingdom
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,"15,818.40",15838,United Kingdom
228042,511465,15044A,PINK PAPER PARASOL,3500,2.55,"8,925.00",18008,United Kingdom
873786,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,5.06,"7,144.72",17450,United Kingdom
578172,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,"6,539.40",15749,United Kingdom
686007,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2.10,"6,539.40",15749,United Kingdom
461644,533027,22086,PAPER CHAIN KIT 50'S CHRISTMAS,835,6.95,"5,803.25",<NA>,United Kingdom
379875,525968,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,1.66,"5,179.20",15838,United Kingdom


Крупное значение само по себе не является ошибкой. Оно может относиться к оптовому покупателю, поэтому выбросы пока сохраняются в данных.

## Сохранение результата

In [22]:
df_clean.to_pickle('online_retail_clean.pkl')

## Итог

В ходе очистки:

- столбцы приведены к подходящим типам;
- текстовые значения очищены от лишних пробелов;
- удалены полные дубликаты;
- добавлены признаки отмены, отрицательного количества и служебной операции;
- рассчитана стоимость позиции;
- строки с пропусками не удалялись без необходимости;
- отдельно выделены обычные продажи и продажи с известным покупателем;
- крупные значения сохранены для дальнейшего анализа.

Очищенная таблица сохранена в файле `online_retail_clean.pkl`.